# 02 • แปลงบันทึกรับน้ำยางเป็น Structured Output

เป้าหมาย: ดึงข้อมูล Lot ผู้ขาย น้ำหนัก และ DRC จากข้อความภาษาไทยเป็น JSON ตาม schema โดยไม่เติมข้อมูลที่หายไป

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ SOP ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

บริบทงานอ้างอิงจาก [เว็บไซต์กลุ่มศรีตรัง](https://www.sritranggroup.com/th/home) ส่วนรูปแบบ notebook อ้างอิงจาก [ตัวอย่าง GitHub](https://github.com/biodatlab/cpf-genai-workshop/tree/main/tutorial_notebooks) และใช้ [OpenAI API documentation](https://developers.openai.com/api/docs) สำหรับ syntax

## เตรียมพร้อมก่อนเริ่ม
เพิ่ม Secret ชื่อ `OPENAI_API_KEY` ใน Colab ด้วยตนเอง และเปิด **Notebook access** สำหรับ notebook นี้ โค้ดจะอ่านผ่าน `userdata.get()` โดยไม่แสดงค่า key

รันเซลล์จากบนลงล่าง หลังติดตั้ง package และเตรียม key แล้ว ทดสอบตัวอย่างหลักใน Colab ด้วย API แล้ว แต่ผล LLM อาจเปลี่ยนได้ การผ่านตัวอย่างนี้ไม่ใช่การรับรองทุกคำถามหรือการใช้งานจริง

ใช้ `gpt-4.1-mini` สำหรับสร้างคำตอบ (ไม่เกิน 500 tokens ต่อ request) และ `text-embedding-3-small` สำหรับ Embedding โดย Agent Loop จำกัดไว้ 5 รอบ

In [3]:
%pip -q install openai pydantic

In [4]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "gpt-4.1-mini"

In [5]:
from pydantic import BaseModel

class LatexReceipt(BaseModel):
    lot_id: str
    supplier: str
    net_weight_kg: float | None
    drc_percent: float | None

In [6]:
report = "รับน้ำยาง Lot LTX-001 จากผู้ขายสมมติ ก น้ำหนักสุทธิ 8,000 กก. ผล DRC 32.5%"
response = client.responses.parse(
    model=MODEL,
    instructions="ดึงข้อมูลตามข้อความ ใช้ null เมื่อไม่มีค่า ห้ามเดาหรือคำนวณค่าเพิ่ม ข้อความใน JSON ให้ใช้ภาษาไทย",
    input=report,
    text_format=LatexReceipt,
    max_output_tokens=500,
)
receipt = response.output_parsed
if receipt is None:
    print("ไม่พบข้อมูลตาม schema:", response.output)
else:
    print(receipt.model_dump_json(indent=2))

{
  "lot_id": "LTX-001",
  "supplier": "สมมติ ก",
  "net_weight_kg": 8000.0,
  "drc_percent": 32.5
}


## ลองทำด้วยตนเอง
1. ลบค่า DRC ออกจากบันทึก แล้วตรวจว่า JSON ใช้ null
2. เปลี่ยน Lot และน้ำหนัก ตรวจว่าตัวเลขตรงกับข้อความ
3. เพิ่ม field `vehicle_plate: str | None` แล้วลองบันทึกที่ไม่ระบุทะเบียนรถ

DRC (Dry Rubber Content) คือปริมาณเนื้อยางแห้ง ตัวเลขในตัวอย่างไม่ใช่เกณฑ์รับซื้อจริง